# Exploratory look at the Xente transactions

Each section is a question the pipeline had to answer, the measurement, and the decision that followed. The model is trained by `python -m credit_risk.train`, not by this notebook. The file is the Xente Fraud Detection Challenge training extract, saved locally as `data/raw/data.csv` and not committed.


## 1. What is in the file, and can it be modeled?

**Question.** Does the extract have a usable schema, a time range, and customers, or is it too broken to build on?


In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from credit_risk.data import load_transactions

raw_path = ROOT / "data" / "raw" / "data.csv"
if not raw_path.is_file():
    raise FileNotFoundError(
        "Save the Xente training file as data/raw/data.csv before running this notebook."
    )

df = load_transactions(raw_path)
print("rows", len(df), "customers", df["CustomerId"].nunique())
print("columns", list(df.columns))
print("nulls", int(df.isna().sum().sum()))
print("from", df["TransactionStartTime"].min())
print("to", df["TransactionStartTime"].max())


rows 95662 customers 3742
columns ['TransactionId', 'BatchId', 'AccountId', 'SubscriptionId', 'CustomerId', 'CurrencyCode', 'CountryCode', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId', 'Amount', 'Value', 'TransactionStartTime', 'PricingStrategy', 'FraudResult']
nulls 0
from 2018-11-15 02:18:49+00:00
to 2019-02-13 10:01:28+00:00


**Finding.** 95,662 transactions, 3,742 customers, 16 columns, no nulls after validation. The window is 15 November 2018 through 13 February 2019, about ninety days.

**Decision.** One transactions table is enough to start. There is no second file and no default column to join.


## 2. Can FraudResult be the credit label?

**Question.** The only outcome in the file is `FraudResult`. Is that a stand-in for default?


In [2]:
fraud = int(df["FraudResult"].sum())
print("fraud rows", fraud)
print("fraud rate", round(fraud / len(df), 5))


fraud rows 193
fraud rate 0.00202


**Finding.** 193 fraudulent rows, a rate of 0.00202. The field is a fraud flag on a transaction, not a missed repayment.

**Decision.** Do not train on `FraudResult`, and do not feed it to the model. It is a different event, and it is too rare to be a stable credit feature.


## 3. Which columns carry no information?

**Question.** Are currency, country, and the two amount fields redundant?


In [3]:
print("currency", sorted(df["CurrencyCode"].unique()))
print("country", [int(value) for value in sorted(df["CountryCode"].unique())])
gap = (df["Value"] - df["Amount"].abs()).abs()
print("rows where Value != abs(Amount)", int((gap > 0).sum()))
print("largest gap", float(gap.max()))


currency ['UGX']
country [256]
rows where Value != abs(Amount) 2565
largest gap 33000.0


**Finding.** Currency is only UGX and country is only 256. `Value` matches `abs(Amount)` on most rows and differs on 2,565 rows, by up to 33,000.

**Decision.** Drop currency and country. Use the sign of `Amount` as the debit flag (positive amounts are debits in this file) and `Value` as the magnitude. Do not treat `Value` as a strict copy of `abs(Amount)`.


## 4. How concentrated is customer activity?

**Question.** Is a customer-level score defined for everyone, or only for customers with a history?


In [4]:
per_customer = df.groupby("CustomerId").size()
print(per_customer.describe().round(1))
print("customers with fewer than 3 transactions", int((per_customer < 3).sum()))
print("share", round(float((per_customer < 3).mean()), 3))


count    3742.0
mean       25.6
std        96.9
min         1.0
25%         2.0
50%         7.0
75%        20.0
max      4091.0
dtype: float64
customers with fewer than 3 transactions 962
share 0.257


**Finding.** The median customer has 7 transactions. The mean is 25.6 because a few customers have hundreds or thousands (the maximum is 4,091). 962 customers, 25.7%, have fewer than 3.

**Decision.** Do not score a customer with fewer than 3 transactions in the window. A one-row history has no recency story, and imputing it would invent one.


## 5. Why is a raw RFM risk score misleading?

**Question.** If high risk is `Recency - Frequency - Monetary` on the raw cluster means, which term actually decides?


In [5]:
grouped = df.groupby("CustomerId")
snapshot = df["TransactionStartTime"].max()
recency = (snapshot - grouped["TransactionStartTime"].max()).dt.total_seconds() / 86400
frequency = grouped["TransactionId"].count()
monetary = grouped["Value"].sum()
scale = pd.DataFrame({"recency_days": recency, "frequency": frequency, "monetary": monetary})
print(scale.agg(["mean", "std"]).round(1))


      recency_days  frequency   monetary
mean          31.0       25.6   253102.5
std           27.1       96.9  2715876.8


**Finding.** The standard deviation of monetary value is about 2.7 million UGX. Recency's is about 27 days, and frequency's is about 97 transactions. A sum of the raw means is a monetary ranking.

**Decision.** Standardize before the risk score. Use rates per day of tenure, then `log1p`, because a few merchants otherwise set the scale. Use recency as a fraction of the observed window, not as absolute days: a later cohort's whole life can be shorter than an early cohort's quiet period, so an absolute cutoff is a state that later customers cannot reach.


## 6. Where does a time-based split fall?

**Question.** If customers are held out by first-seen time, what dates are validation and test, and is the test cohort large enough to report?


In [6]:
from credit_risk.split import assign_cohorts

cohorts = assign_cohorts(df, val_fraction=0.2, test_fraction=0.2)
print("customers", len(cohorts.fit_ids), len(cohorts.val_ids), len(cohorts.test_ids))
print("validation starts", cohorts.val_start)
print("test starts", cohorts.test_start)
print("file ends", cohorts.data_end)


customers 2246 748 748
validation starts 2019-01-01 22:04:20+00:00
test starts 2019-01-23 09:32:57+00:00
file ends 2019-02-13 10:01:28+00:00


**Finding.** Ordering 3,742 customers by their first transaction puts 2,246 in fit, 748 in validation, and 748 in test. Validation starts on 1 January 2019. The test cohort starts on 23 January 2019. The file ends on 13 February 2019. After the minimum of 3 transactions, the training run scores 1,654, 461, and 560 of those customers. The segment rate is about 0.21, 0.27, and 0.22, so the label is not confined to the early cohort.

**Decision.** Report metrics only on the test cohort. Fit the segmenter, the weight-of-evidence edges, and the level vocabulary on the fit cohort. Choose the model and the cutoff on validation. The held-out numbers are in `reports/`.


## 7. What does the model get to see?

**Question.** Once the label is a cluster of recency fraction and log activity rates, which customer fields are still safe to use?

**Finding.** Shares, entropy of a mix, the sign of the amount, and the shape of the clock do not restate the three label inputs. Unique values divided by the transaction count does: it moves with the inverse of frequency. So does the mean gap between transactions.

**Decision.** The training matrix keeps the mix and the shape, and the training script refuses the label columns. Whether that matrix actually tracks the segment is a measured result, not an assumption. On this file the served forest has a test ROC-AUC of 0.785, and a model that is allowed to see the label inputs scores 0.9996. The second number is the leakage diagnostic. It is not the model.
